# Author: Michael Chen (mwc2150@columbia.edu)

# 0. Introduction / Set-up

This notebook handles the Named Entity Recognition (NER) step for the fashion brand Everlane, which aims to answer the question: "Has Everlane's sustainability messaging become more concrete over time, or has it diverged from public perceptions of ethical practice?"

3 distinct sources were scraped to construct a multi-perspective corpus:
1. reddit_comments.csv and reddit_posts.csv - 4,600 Reddit posts and 12,400 comments covering consumer sentiment,
2. gdelt_articles_usable.csv - 28,400 lines from over 1,000 news articles capturing media coverage,
3. everlane_web.csv - nearly 1,400 lines scraped directly from Everlane's website. 

After splitting these sources into a sentence-level corpus, I will now attempt to use NER to extract places, people, organizations, and other revelant topics / miscellanea which may include:
- Actors - i.e. Everlane, Shein
- Sources - Reddit, New York Times,etc.
- Time Periods - Any months from the years 2016-2026 (Hopefully, through this process we can set some implicitly-significant criteria for what time periods we want to hone in on for this deepdive. )

 Additionally, I will attempt to pull multiple BERT-based models from HuggingFace and evaluate their appropriateness/efficacy given our social media, news, and point-of-sale data. Overall, the notebook serves a dual purpose: I want to use NER to get observations that stand on their own from an analytical standpoint, and I also want to build towards outputs that feed into an interparty-sentiment step later down the line. 

# 1. Configurations

I am going to first start by documenting the paths, candidate model ID's, batch size, confidence threshold, and seed for repeatability. None of this is by any means final.

In [1]:
import os

# On Colab (incl. the VS Code Colab extension), get the repo so ../data paths work like they do locally
# Allows us to take advantage of Colab's free GPU/TPU resources while still being able to run the notebooks locally
if os.path.exists("/content"):
    !git clone -q --depth 1 https://github.com/mwc2150/everlane-sustainability-analysis.git /content/repo 2>/dev/null || git -C /content/repo pull -q
    %cd /content/repo/notebooks

import sys
print("Running on:", "Colab" if os.path.exists("/content") else "local")
print("Python:", sys.executable)

Running on: local
Python: c:\Users\micha\everlane_deepdive\.venv\Scripts\python.exe


In [2]:
import json, random
from pathlib import Path
from datetime import datetime

import numpy as np
import torch


In [3]:
# Paths 
# Loading in the sentences from the CSVs in data/sentences, and saving the NER results to data/ner/<run_id>
PROJECT_ROOT = Path.cwd().parent
SENT_DIR     = PROJECT_ROOT / "data" / "sentences"
INPUT_FILES  = {
    "everlane_web":    SENT_DIR / "everlane_web_sentences.csv",
    "gdelt_articles":  SENT_DIR / "gdelt_articles_sentences.csv",
    "reddit_posts":    SENT_DIR / "reddit_posts_sentences.csv",
    "reddit_comments": SENT_DIR / "reddit_comments_sentences.csv",
}
RUN_ID  = datetime.now().strftime("%Y%m%d_%H%M")
OUT_DIR = PROJECT_ROOT / "data" / "ner" / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
# NER model (id, revision). The revision is pinned to a commit hash so reruns use identical weights.
# dslim/bert-base-NER is BERT-base fine-tuned on CoNLL-2003 (Reuters newswire, 1996-97), labels PER / ORG / LOC / MISC.
NER_MODEL   = ("dslim/bert-base-NER", "d1a3e8f13f8c3566299d95fcfc9a8d2382a9affc")
AGGREGATION = "first"    # word-level merging; "simple" split words on dev (e.g. "Lea" + "ther Working Group")

# Inference
BATCH_SIZE           = 128 if torch.cuda.is_available() else 16  # T4 on Colab vs local CPU
MAX_LENGTH           = 256    # tokens; longer sentences are truncated
CONFIDENCE_THRESHOLD = 0.80   # drop entities scored below this
DEV_SAMPLE_N         = None   # e.g. 2_000 for a quick sample; None runs the full corpus

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else -1  # HF pipeline convention

# Save the config with this run's outputs
config = {k: (str(v) if isinstance(v, Path) else v)
          for k, v in globals().items() if k.isupper()}
config["INPUT_FILES"] = {k: str(v) for k, v in INPUT_FILES.items()}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2))
print(json.dumps(config, indent=2))

{
  "PROJECT_ROOT": "C:\\Users\\micha\\everlane_deepdive",
  "SENT_DIR": "C:\\Users\\micha\\everlane_deepdive\\data\\sentences",
  "INPUT_FILES": {
    "everlane_web": "C:\\Users\\micha\\everlane_deepdive\\data\\sentences\\everlane_web_sentences.csv",
    "gdelt_articles": "C:\\Users\\micha\\everlane_deepdive\\data\\sentences\\gdelt_articles_sentences.csv",
    "reddit_posts": "C:\\Users\\micha\\everlane_deepdive\\data\\sentences\\reddit_posts_sentences.csv",
    "reddit_comments": "C:\\Users\\micha\\everlane_deepdive\\data\\sentences\\reddit_comments_sentences.csv"
  },
  "RUN_ID": "20260927_0327",
  "OUT_DIR": "C:\\Users\\micha\\everlane_deepdive\\data\\ner\\20260927_0327",
  "NER_MODEL": [
    "dslim/bert-base-NER",
    "d1a3e8f13f8c3566299d95fcfc9a8d2382a9affc"
  ],
  "AGGREGATION": "first",
  "BATCH_SIZE": 16,
  "MAX_LENGTH": 256,
  "CONFIDENCE_THRESHOLD": 0.8,
  "DEV_SAMPLE_N": null,
  "SEED": 42,
  "DEVICE": -1
}


# 2. Data Loading and Preprocessing

Next, I'm going to create and check a common schema for analyzing our 3 data sources such that we drop nulls and duplicates, print counts by source and time period, and handle any last syntax/formatting gaps that may hinder the model from doing proper NER. We also have to be so careful as to not programatically drop any data (especially from our Reddit crawl) that could be perceived as noise due to sentence length. Mentions that capture outrage, enjoyment, or otherwise neutral sentiments-- however short-- must be preserved.

## *Couple things to note of below:

### Double checking our deduplication step 
 - Especially for the Reddit posts/comments, any false removals due to links or images being processed improperly by pandas had to be resolved with an extra document id check
 
### Peculiarities with using the dataset from GDELT
- The date range can sometimes error out. 1969-12-31 19:00 is a placeholder, not a real publication date. It's Unix time zero (midnight on 1 Jan 1970 UTC) displayed in US Eastern time, which the scraper writes when it can't find a date. 
- This affects far more than it first looked: 385 of the 1,015 articles (about a third of GDELT sentences) have no scraped publish date. Rather than re-scraping, which could change article text and break the frozen eval set, missing dates are filled with the date GDELT first saw each article, taken from the saved GDELT search results. For articles that do have a publish date, the two agree within a day 95% of the time. The `date_source` column records which date each sentence uses (`published`, `gdelt_first_seen`, or `scrape_date` for the Everlane site, whose pages have no publish date).
- GDELT also returns foreign-language coverage (mostly German, Spanish, French and Italian reporting on the Shein acquisition). dslim only reads English, so articles whose overall language isn't English are dropped. Language is detected per whole article with `lingua`, because detection on single sentences misfires on short English text ("But Everlane jeans are my favorite!" comes out as Swedish). Reddit and the Everlane site are English-language sources and aren't filtered.

In [5]:
import re, html, unicodedata
import pandas as pd

# Load & harmonize to a schema
RENAME = {"publish_date": "date"}
DOC_ID = {"everlane_web": "url", "gdelt_articles": "url",
          "reddit_posts": "post_id", "reddit_comments": "comment_id"}

frames = []
for src, path in INPUT_FILES.items():
    d = pd.read_csv(path).rename(columns=RENAME)
    d["source"] = src
    d["doc_id"] = d[DOC_ID[src]].astype("string")
    if src == "reddit_posts":  # link/image posts have no post_id; url + timestamp keeps cross-posts distinct
        d["doc_id"] = d["doc_id"].fillna(d["url"] + "@" + d["date"])
    frames.append(d)
raw = pd.concat(frames, ignore_index=True)

raw["date_raw"] = raw["date"]  # keep the original string for traceability
raw["date"] = pd.to_datetime(raw["date"].str[:10], errors="coerce")
raw.loc[raw["date"].dt.year < 2000, "date"] = pd.NaT  # 1969-12-31 = epoch-zero placeholder, treat as missing

# GDELT articles without a scraped publish date get the date GDELT first saw them (saved from the GDELT searches);
# for articles that do have a publish date, the two agree within a day 95% of the time
INTERMEDIATE = PROJECT_ROOT / "data" / "intermediate"
seen = pd.concat([
    pd.read_csv(INTERMEDIATE / "gdelt_historical.csv", usecols=["DocumentIdentifier", "date"], dtype=str)
      .rename(columns={"DocumentIdentifier": "url", "date": "seen"}),
    pd.read_csv(INTERMEDIATE / "gdelt_news.csv", usecols=["url", "seendate"], dtype=str)
      .rename(columns={"seendate": "seen"}),
])
seen = pd.to_datetime(seen["seen"].str[:8], format="%Y%m%d").groupby(seen["url"]).min()

raw["date_source"] = np.where(raw["source"].eq("everlane_web"), "scrape_date", "published")  # site pages carry no publish date
raw.loc[raw["date"].isna(), "date_source"] = None
backfill = raw["source"].eq("gdelt_articles") & raw["date"].isna()
raw.loc[backfill, "date"] = raw.loc[backfill, "url"].map(seen)
raw.loc[backfill & raw["date"].notna(), "date_source"] = "gdelt_first_seen"
raw["period"] = raw["date"].dt.to_period("Q").astype("string")  # quarters; swap for "Y" if too sparse

# Nulls & duplicates (log every drop)
log = {"loaded": raw.groupby("source").size()}
df = raw.dropna(subset=["sentence"])
df = df[df["sentence"].str.strip().ne("")]
log["after_null_text"] = df.groupby("source").size()
df = df.drop_duplicates(subset=["source", "doc_id", "sentence_id"])
log["after_dedup"] = df.groupby("source").size()
df["is_dup_text"] = df.duplicated(subset=["sentence"], keep="first")  # flag, don't drop

# Light normalization for NER (keep the original)
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MD_RE  = re.compile(r"[*_>#`]+|\[([^\]]+)\]\([^)]+\)")

def clean(s: str) -> str:
    s = unicodedata.normalize("NFKC", html.unescape(s))
    s = URL_RE.sub(" ", s)
    s = MD_RE.sub(lambda m: m.group(1) or " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    if s.isupper() and len(s.split()) <= 8:   # ALL-CAPS headers confuse cased models
        s = s.title()
    return s

df["text"] = df["sentence"].astype(str).map(clean)
JUNK = {"[deleted]", "[removed]"}
has_word = df["text"].str.contains(r"[A-Za-z]{2,}")
df = df[has_word & ~df["text"].str.lower().isin(JUNK)]
log["after_junk_filter"] = df.groupby("source").size()

# Language: GDELT pulls in foreign-language coverage that dslim (English-only) can't read.
# Detected per whole article, since sentence-level detection misfires on short English text;
# Reddit and the Everlane site are English-language sources and are not filtered.
from lingua import LanguageDetectorBuilder
detector = LanguageDetectorBuilder.from_all_languages().with_preloaded_language_models().build()
is_gdelt = df["source"].eq("gdelt_articles")
articles = df[is_gdelt].groupby("doc_id")["text"].agg(" ".join)
langs = detector.detect_languages_in_parallel_of(articles.tolist())
doc_lang = pd.Series([l.iso_code_639_1.name if l else "UNKNOWN" for l in langs], index=articles.index)
df["doc_lang"] = df["doc_id"].map(doc_lang).where(is_gdelt, "EN")
display(df[is_gdelt].drop_duplicates("doc_id")["doc_lang"].value_counts().rename("GDELT articles by language").to_frame().T)
df = df[df["doc_lang"].eq("EN")]
log["after_language_filter"] = df.groupby("source").size()
df["uid"] = df["source"] + ":" + df["doc_id"] + ":" + df["sentence_id"].astype(str)
raw.loc[~raw.index.isin(df.index), ["source", "doc_id", "sentence_id", "sentence"]].to_csv(OUT_DIR / "dropped_sentences.csv", index=False)

# Checks & reporting
assert df["doc_id"].notna().all()
assert df["uid"].is_unique
assert df["text"].str.len().gt(0).all()
drop_log = pd.DataFrame(log).fillna(0).astype(int)
drop_log.to_csv(OUT_DIR / "load_log.csv")
display(drop_log)
display(df.groupby("source")["date"].agg(missing=lambda s: s.isna().sum(), min="min", max="max"))
display(pd.crosstab(df["source"], df["date_source"].fillna("missing")))
display(pd.crosstab(df["period"], df["source"]).sort_index())

doc_lang,EN,DE,ES,FR,IT,PT,VI,RU,YO,JA,NL,PL,EL,HR,RO
GDELT articles by language,955,14,11,10,7,5,2,2,2,1,1,1,1,1,1


,loaded,after_null_text,after_dedup,after_junk_filter,after_language_filter
source,,,,,
everlane_web,1666,1666,1666,1607,1607
gdelt_articles,25695,25695,25695,25488,24200
reddit_comments,20787,20787,20787,19948,19948
reddit_posts,6162,6162,6162,5661,5661


,missing,min,max
source,,,
everlane_web,0,2026-05-19,2026-05-19
gdelt_articles,0,2016-01-06,2026-05-27
reddit_comments,0,2013-01-30,2026-05-19
reddit_posts,0,2013-01-30,2026-05-18


date_source,gdelt_first_seen,published,scrape_date
source,,,
everlane_web,0,0,1607
gdelt_articles,8374,15826,0
reddit_comments,0,19948,0
reddit_posts,0,5661,0


source,everlane_web,gdelt_articles,reddit_comments,reddit_posts
period,,,,
2013Q1,0,0,290,96
2013Q2,0,0,113,54
2013Q3,0,0,281,238
2013Q4,0,0,153,107
2014Q1,0,0,504,138
2014Q2,0,0,143,82
2014Q3,0,0,328,99
2014Q4,0,0,172,72
2015Q1,0,0,215,96


# 3. Evaluation Set

I am next going to preserve some sentences that stratified across sources and periods and label them before running any models. One of the key things here to look out for is making sure that I'm not oversampling from years where only a select few sentences could be chosen. I also don't want to give every year equal weight, as that would make the earlier years get dissproportionately more share as heavy recent ones. Instead of splitting on raw periods (years), I am going to simply sample from within each source, split it at random, and report the years that it happens to cover. 

In [6]:
# 3. Evaluation set: sampled BEFORE any model runs, frozen at a fixed path
EVAL_DIR  = PROJECT_ROOT / "data" / "eval"  # shared with sentiment.ipynb
EVAL_PATH = EVAL_DIR / "eval_set.csv"
N_PER_SOURCE, N_DEV_PER_SOURCE = 38, 13   # ~150 total, ~50 dev / ~100 test

if EVAL_PATH.exists():
    eval_df = pd.read_csv(EVAL_PATH, keep_default_na=False)
    print(f"Loaded frozen eval set ({len(eval_df)} rows) - not resampling")
else:
    EVAL_DIR.mkdir(parents=True, exist_ok=True)
    pool = df[~df["is_dup_text"]].copy()
    pool["year"] = pool["date"].dt.year.astype("Int64").astype("string").fillna("unknown")

    # Equal n per source so GDELT doesn't dominate; simple random sample within each source
    eval_df = pool.groupby("source").sample(n=N_PER_SOURCE, random_state=SEED)

    # Random dev/test split within each source (dev is for threshold tuning, test is reported)
    dev_idx = eval_df.groupby("source").sample(n=N_DEV_PER_SOURCE, random_state=SEED).index
    eval_df["split"] = np.where(eval_df.index.isin(dev_idx), "dev", "test")

    # Shuffle row order so labeling isn't done source-by-source (avoids drift in how you label)
    eval_df = eval_df[["uid", "source", "year", "split", "text"]].sample(frac=1, random_state=SEED)
    eval_df["gold_entities"] = ""   # fill as "Everlane|ORG; Peru|LOC"; leave blank if none
    eval_df["notes"] = ""
    eval_df.to_csv(EVAL_PATH, index=False)
    print(f"Wrote {len(eval_df)} rows to {EVAL_PATH}. Label them before running any model.")

display(pd.crosstab(eval_df["split"], eval_df["source"], margins=True))  # 13 dev / 25 test per source
display(pd.crosstab(eval_df["year"], eval_df["source"]))               # year coverage, reported not enforced

Loaded frozen eval set (152 rows) - not resampling


source,everlane_web,gdelt_articles,reddit_comments,reddit_posts,All
split,,,,,
dev,13,13,13,13,52
test,25,25,25,25,100
All,38,38,38,38,152


source,everlane_web,gdelt_articles,reddit_comments,reddit_posts
year,,,,
2013,0,0,1,3
2014,0,0,3,2
2015,0,0,2,1
2016,0,2,2,4
2017,0,2,8,2
2018,0,3,3,1
2019,0,5,6,7
2020,0,1,8,5
2021,0,2,0,9


### 3a. Gold entity labels

Next, I hand-label every entity in the 152 frozen sentences before running dslim, so the model is scored against labels written without seeing its output. Entities are written exactly as they appear in the sentence, as `Name|TYPE` separated by `;`, using the four types dslim outputs:

- **PER**: people ("Baskauskas", "Laufey")
- **ORG**: companies, brands, publications, NGOs, subreddits ("Everlane", "KOTN", "Leather Working Group", "LWG", "FFA")
- **LOC**: countries, cities, regions, factory locations ("Peru", "San Francisco", "Thai Binh")
- **MISC**: named events, campaigns, slogans, nationalities ("Radical Transparency", "Italian")

Edge cases:
- Brands are ORG even when plural or used as a product ("Timbuk2s" = ORG); abbreviations are labeled as written ("CK" = ORG).
- Generic product words are not entities ("Espadrilles", "t-shirts").
- An entity mentioned twice in one sentence is listed once.
- Named buildings and factory complexes are ORG ("Rana Plaza"). This rule was written down after the first dslim run to record a label I had already made; no label was changed.
- A sentence with no entities is confirmed with Enter, which writes "no entities" to `notes`.

The labeling cell checks that each name appears verbatim in the sentence and that the type is valid, saves after every answer, and resumes where it left off. It has to run locally.

In [7]:
# Label entities one sentence at a time:  Everlane|ORG; Peru|LOC
# Enter = no entities, s = skip, q = quit and resume later
TYPES = {"PER", "ORG", "LOC", "MISC"}

def save_label(uid, col, value, append=False):
    # Re-read the file before every write: ner.ipynb and sentiment.ipynb both label this file,
    # so writing back an in-memory copy would erase the other notebook's labels
    d = pd.read_csv(EVAL_PATH, dtype=str, keep_default_na=False)
    row = d["uid"] == uid
    if append:
        value = "; ".join(filter(None, [d.loc[row, col].iat[0], value]))
    d.loc[row, col] = value
    d.to_csv(EVAL_PATH, index=False)


def parse_entities(answer):
    ents = [x.strip().rsplit("|", 1) for x in answer.split(";") if x.strip()]
    return [(e[0].strip(), e[1].strip().upper()) if len(e) == 2 else (e[0], None) for e in ents]

def check(text, answer):
    errors = []
    for name, typ in parse_entities(answer):
        if typ not in TYPES:
            errors.append(f"'{name}': type must be one of {sorted(TYPES)}")
        elif name not in text:
            errors.append(f"'{name}' not found verbatim in the sentence")
    return errors

eval_df = pd.read_csv(EVAL_PATH, dtype=str, keep_default_na=False)
done = eval_df["gold_entities"].ne("") | eval_df["notes"].str.contains("no entities")
todo = eval_df[~done]
print(f"{len(todo)} left to label")

for i, row in enumerate(todo.itertuples(), 1):
    print(f"\n[{i}/{len(todo)}] {row.text}")
    answer = input("Name|TYPE; ...  (Enter = none, s = skip, q = quit) > ").strip()
    while answer not in {"s", "q"} and (errors := check(row.text, answer)):
        print("  " + "\n  ".join(errors))
        answer = input("  try again > ").strip()
    if answer == "q":
        break
    if answer == "s":
        continue
    if answer:
        save_label(row.uid, "gold_entities", "; ".join(f"{n}|{t}" for n, t in dict.fromkeys(parse_entities(answer))))
    else:
        save_label(row.uid, "notes", "no entities", append=True)

0 left to label


In [8]:
# Progress: labeled rows and entity counts by type
eval_df = pd.read_csv(EVAL_PATH, dtype=str, keep_default_na=False)
done = eval_df["gold_entities"].ne("") | eval_df["notes"].str.contains("no entities")
print(f"Labeled {done.sum()} / {len(eval_df)}")

# One row per entity (reset_index: sentences with several entities would otherwise repeat an index label)
ents = (eval_df.assign(entity=eval_df["gold_entities"].str.split("; "))
               .explode("entity").query("entity != ''").reset_index(drop=True))
ents["type"] = ents["entity"].str.rsplit("|", n=1).str[1]
display(pd.crosstab(ents["source"], ents["type"], margins=True))

Labeled 152 / 152


type,LOC,MISC,ORG,PER,All
source,,,,,
everlane_web,9,0,8,2,19
gdelt_articles,2,1,18,3,24
reddit_comments,1,0,13,0,14
reddit_posts,7,0,15,0,22
All,19,1,54,5,79


# 4. Baseline: dslim/bert-base-NER on the dev split

I'm using `dslim/bert-base-NER` because it is a widely used, documented baseline (BERT-base fine-tuned on CoNLL-2003, the standard NER benchmark), which makes its errors easy to interpret; I considered `dslim/bert-large-NER` and `Jean-Baptiste/roberta-large-ner-english`, but both are trained on the same CoNLL-2003 news data, so they share its blind spots at roughly three times the size. Its known limits (news-trained, so lowercase brand mentions, product names and subreddits are often missed) will be covered by a gazetteer of known names; this section measures how large those gaps are first.

Only the 52 dev sentences are run here. I can look at every error on dev and tune the confidence threshold on it; the 100 test sentences stay untouched until the setup is final, and are scored once.

In [9]:
from transformers import pipeline

eval_df = pd.read_csv(EVAL_PATH, dtype=str, keep_default_na=False)
assert (eval_df["gold_entities"].ne("") | eval_df["notes"].str.contains("no entities")).all(), \
    "Finish labeling the eval set before running any model"
in_scope = eval_df["uid"].isin(df["uid"])   # rows removed in preprocessing (non-English articles) are out of scope
print(f"{(~in_scope).sum()} eval rows out of scope:", eval_df.loc[~in_scope, "split"].value_counts().to_dict())
dev = eval_df[eval_df["split"].eq("dev") & in_scope]   # test stays untouched until the setup is final

model_id, revision = NER_MODEL
ner = pipeline("token-classification", model=model_id, revision=revision,
               aggregation_strategy=AGGREGATION, device=DEVICE)
outputs = ner(dev["text"].tolist(), batch_size=BATCH_SIZE)

# Keep every prediction with its score (no threshold yet) so the threshold can be tuned below
preds = pd.DataFrame(
    [{"uid": uid, "entity": text[e["start"]:e["end"]], "type": e["entity_group"], "score": float(e["score"]),
      "start": e["start"], "end": e["end"]}
     for uid, text, ents in zip(dev["uid"], dev["text"], outputs) for e in ents],
    columns=["uid", "entity", "type", "score", "start", "end"])
preds.to_csv(OUT_DIR / "dev_predictions.csv", index=False)
print(f"{len(preds)} predicted entities on {len(dev)} dev sentences")

3 eval rows out of scope: {'dev': 3}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


29 predicted entities on 49 dev sentences


In [10]:
# Scoring: an entity counts as correct only if its exact text AND type match a gold label in the same sentence
ENT_TYPES = ["PER", "ORG", "LOC", "MISC"]

def gold_set(s):   # "Everlane|ORG; Peru|LOC" -> {("Everlane", "ORG"), ("Peru", "LOC")}
    return {tuple(x.rsplit("|", 1)) for x in s.split("; ") if x}

def pred_sets(threshold, predictions=None):
    predictions = preds if predictions is None else predictions
    kept = predictions[predictions["score"] >= threshold]
    return {uid: set(zip(g["entity"], g["type"])) for uid, g in kept.groupby("uid")}

def prf(gold, pred, typed=True, only=None):
    tp = fp = fn = 0
    for uid, g in gold.items():
        q = pred.get(uid, set())
        if only:                      # restrict to one entity type
            g, q = {e for e in g if e[1] == only}, {e for e in q if e[1] == only}
        if not typed:                 # compare text only, ignoring the type
            g, q = {e[0] for e in g}, {e[0] for e in q}
        tp += len(g & q); fp += len(q - g); fn += len(g - q)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec  = tp / (tp + fn) if tp + fn else 0.0
    f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"precision": prec, "recall": rec, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

gold = dict(zip(dev["uid"], dev["gold_entities"].map(gold_set)))

# 1. Threshold sweep: pick CONFIDENCE_THRESHOLD here (dev only)
sweep = pd.DataFrame({t: prf(gold, pred_sets(t)) for t in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]}).T
sweep.index.name = "threshold"
display(sweep.round(3))

# 2. At the configured threshold: overall, text-only, and per type
pred = pred_sets(CONFIDENCE_THRESHOLD)
summary = pd.DataFrame({"text + type": prf(gold, pred),
                        "text only":   prf(gold, pred, typed=False),
                        **{t: prf(gold, pred, only=t) for t in ENT_TYPES}}).T
display(summary.round(3))

# 3. Every sentence where dslim and the gold labels disagree
fmt = lambda ents: "; ".join(f"{e}|{t}" for e, t in sorted(ents))
errors = pd.DataFrame(
    [{"source": src, "text": text, "missed": fmt(gold[uid] - pred.get(uid, set())),
      "extra": fmt(pred.get(uid, set()) - gold[uid])}
     for uid, src, text in zip(dev["uid"], dev["source"], dev["text"]) if gold[uid] != pred.get(uid, set())])
with pd.option_context("display.max_colwidth", None):
    display(errors)

sweep.to_csv(OUT_DIR / "dev_threshold_sweep.csv"); summary.to_csv(OUT_DIR / "dev_summary.csv")
errors.to_csv(OUT_DIR / "dev_errors.csv", index=False)

,precision,recall,f1,tp,fp,fn
threshold,,,,,,
0.00,0.621,0.581,0.600,18.0,11.0,13.0
0.50,0.654,0.548,0.596,17.0,9.0,14.0
0.60,0.708,0.548,0.618,17.0,7.0,14.0
0.70,0.708,0.548,0.618,17.0,7.0,14.0
0.80,0.727,0.516,0.604,16.0,6.0,15.0
0.90,0.762,0.516,0.615,16.0,5.0,15.0
0.95,0.800,0.516,0.627,16.0,4.0,15.0


,precision,recall,f1,tp,fp,fn
text + type,0.727,0.516,0.604,16.0,6.0,15.0
text only,0.955,0.677,0.792,21.0,1.0,10.0
PER,0.400,0.667,0.500,2.0,3.0,1.0
ORG,1.000,0.294,0.455,5.0,0.0,12.0
LOC,1.000,0.900,0.947,9.0,0.0,1.0
MISC,0.000,0.000,0.000,0.0,3.0,1.0


,source,text,missed,extra
0,gdelt_articles,"Under the motto “Radical Transparency,” the brand gave customers insight into the origin of the raw materials it uses, as well as the design and production processes and logistics.",Radical Transparency|MISC,
1,everlane_web,Thai Binh,Thai Binh|LOC,
2,reddit_posts,"I was a little hesitant to consider them, well, because 9 out of 10 backpacks I see here (San Francisco) are Timbuk2s.",Timbuk2s|ORG,Timbuk2s|MISC
3,everlane_web,We ended up choosing two distinct fill materials—PrimaLoft® sheeting and insulation—and used water-wicking poly-nylon on the outside.,PrimaLoft|ORG,
4,everlane_web,"Originally from Naples, Pasquale and Aimone began as stitchers and became passionate about the shoe manufacturing process, eventually moving to Pisa to open their own business.",Aimone|PER,
5,everlane_web,"Following the Rana Plaza factory collapse in 2013, we made a commitment to transparently disclose the name and location of our factory partners.",Rana Plaza|ORG,
6,reddit_posts,"Thank you so much for the recommendations- I was already planning an Everlane order, so I'll cut a little somewhere else to make up the gap.",Everlane|ORG,
7,reddit_comments,Alex Mill!,Alex Mill|ORG,Alex Mill|PER
8,gdelt_articles,"Castaner canvas wedge espadrilles, £80, Net-A-Porter – buy now",Castaner|ORG,
9,reddit_posts,"All of Olori’s handbags are made by artisans and women-owned businesses in Nigeria, AND each bag pays one month of tuition fees for an underprivileged girl in Africa.",Olori|ORG,Olori|PER


# 5. Gazetteer

dslim is trained on 1990s newswire, so it misses brand names, especially lowercase ones on Reddit ("zara", "shein"), and sometimes tags brands as people. A gazetteer (a hand-curated list of known names matched as text) covers that gap. The list is in `data/gazetteer/gazetteer.csv`, and it was built independently of the dev errors so the dev score isn't inflated:

- **Candidates:** capitalized names that appear mid-sentence at least 15 times in the corpus, with all 152 eval sentences removed, plus the Everlane slogans named in the README ("Radical Transparency", "Clean Luxury").
- **Kept:** brands, retailers, platforms, investors and factories (ORG), and named slogans and events (MISC). People, places and publications are left to dslim, which already handles them well.
- **Case:** names match in any case ("zara" = Zara), except names that are also everyday words ("Gap", "Target", "Theory", "Loft", "Apple"), which only match when capitalized. "Express" and "Vince" were dropped because they matched "Panda Express", "express shipping" and first names.

Where a gazetteer match overlaps a dslim entity, the gazetteer wins (its span and type); dslim entities that don't overlap are kept.

In [11]:
GAZ_PATH = PROJECT_ROOT / "data" / "gazetteer" / "gazetteer.csv"
gaz = pd.read_csv(GAZ_PATH)

def gaz_pattern(name, case_sensitive):
    body = re.escape(name).replace("'", "['’]")        # match straight and curly apostrophes ("Levi’s")
    return re.compile(rf"(?<![\w&]){body}s?(?![\w&])",  # whole words only; allow a plural "s"
                      0 if case_sensitive else re.IGNORECASE)

GAZ = [(gaz_pattern(r.name, r.case_sensitive), r.type, r.name) for r in gaz.itertuples()]

def gazetteer_spans(text):
    hits = sorted(((m.start(), m.end(), typ, name) for pat, typ, name in GAZ for m in pat.finditer(text)),
                  key=lambda h: (h[0], h[0] - h[1]))     # at each position, longest match first ("Nordstrom Rack" over "Nordstrom")
    spans, last_end = [], -1
    for s, e, t, name in hits:
        if s >= last_end:
            spans.append((s, e, t, name)); last_end = e
    return spans

def merge_entities(text, model_ents):
    # model_ents: [(start, end, type, score)] already above the threshold
    # returns [(start, end, type, score, origin, canonical)]; canonical is the gazetteer entry ("zara" -> "Zara")
    g = gazetteer_spans(text)
    m = [x for x in model_ents if not any(x[0] < y[1] and y[0] < x[1] for y in g)]   # gazetteer wins on overlap
    return ([(s, e, t, None, "gazetteer", name) for s, e, t, name in g] +
            [(s, e, t, sc, "dslim", text[s:e]) for s, e, t, sc in m])

def merged_sets(threshold, rows=None, predictions=None):
    rows = dev if rows is None else rows
    predictions = preds if predictions is None else predictions
    kept = predictions[predictions["score"] >= threshold]
    by_uid = {uid: list(zip(g["start"], g["end"], g["type"], g["score"])) for uid, g in kept.groupby("uid")}
    return {uid: {(text[s:e], t) for s, e, t, *_ in merge_entities(text, by_uid.get(uid, []))}
            for uid, text in zip(rows["uid"], rows["text"])}

merged = merged_sets(CONFIDENCE_THRESHOLD)
compare = pd.DataFrame({"dslim": prf(gold, pred), "dslim + gazetteer": prf(gold, merged),
                        **{f"dslim + gazetteer: {t}": prf(gold, merged, only=t) for t in ENT_TYPES}}).T
display(compare.round(3))

merged_errors = pd.DataFrame(
    [{"source": src, "text": text, "missed": fmt(gold[uid] - merged[uid]), "extra": fmt(merged[uid] - gold[uid])}
     for uid, src, text in zip(dev["uid"], dev["source"], dev["text"]) if gold[uid] != merged[uid]])
with pd.option_context("display.max_colwidth", None):
    display(merged_errors)

compare.to_csv(OUT_DIR / "dev_gazetteer_compare.csv"); merged_errors.to_csv(OUT_DIR / "dev_gazetteer_errors.csv", index=False)

,precision,recall,f1,tp,fp,fn
dslim,0.727,0.516,0.604,16.0,6.0,15.0
dslim + gazetteer,0.815,0.710,0.759,22.0,5.0,9.0
dslim + gazetteer: PER,0.400,0.667,0.500,2.0,3.0,1.0
dslim + gazetteer: ORG,1.000,0.588,0.741,10.0,0.0,7.0
dslim + gazetteer: LOC,1.000,0.900,0.947,9.0,0.0,1.0
dslim + gazetteer: MISC,0.333,1.000,0.500,1.0,2.0,0.0


,source,text,missed,extra
0,everlane_web,Thai Binh,Thai Binh|LOC,
1,reddit_posts,"I was a little hesitant to consider them, well, because 9 out of 10 backpacks I see here (San Francisco) are Timbuk2s.",Timbuk2s|ORG,Timbuk2s|MISC
2,everlane_web,We ended up choosing two distinct fill materials—PrimaLoft® sheeting and insulation—and used water-wicking poly-nylon on the outside.,PrimaLoft|ORG,
3,everlane_web,"Originally from Naples, Pasquale and Aimone began as stitchers and became passionate about the shoe manufacturing process, eventually moving to Pisa to open their own business.",Aimone|PER,
4,everlane_web,"Following the Rana Plaza factory collapse in 2013, we made a commitment to transparently disclose the name and location of our factory partners.",Rana Plaza|ORG,
5,reddit_comments,Alex Mill!,Alex Mill|ORG,Alex Mill|PER
6,gdelt_articles,"Castaner canvas wedge espadrilles, £80, Net-A-Porter – buy now",Castaner|ORG,
7,reddit_posts,"All of Olori’s handbags are made by artisans and women-owned businesses in Nigeria, AND each bag pays one month of tuition fees for an underprivileged girl in Africa.",Olori|ORG,Olori|PER
8,gdelt_articles,At least they were cheaper than Jesse Kamm’s.,Jesse Kamm|ORG,Jesse Kamm|PER
9,reddit_posts,"After reading several blogs, watching hours of YouTube videos and scouring numerous Pinterest boards, I realized that a simpler, classic wardrobe was more my style.",,YouTube|MISC


# 6. Test Set (Frozen Setup, Scored Once)

The setup was frozen on 2026-09-27, after tuning on dev and before looking at any test result:

| Part | Frozen value |
|---|---|
| Model | `dslim/bert-base-NER`, pinned revision |
| Word merging | `first` |
| Confidence threshold | 0.80 |
| Gazetteer | `data/gazetteer/gazetteer.csv` (76 entries); gazetteer wins on overlap |
| Language filter | GDELT articles must be English as a whole |
| Labeling rules | Section 3a |

The 100 test sentences are scored with exactly this setup. The first run writes a fingerprint of the setup to `data/eval/frozen_setup.json`; if any of it changes afterwards, this cell stops with an error, because the test score would then no longer describe the setup being used. Test errors can be read for the write-up, but nothing is changed because of them.

The 95% confidence intervals come from resampling the test sentences with replacement 2,000 times.

In [12]:
import hashlib

# Fingerprint of the frozen setup; a later mismatch means the setup changed after test was scored
SETUP = {"model": list(NER_MODEL), "aggregation": AGGREGATION, "threshold": CONFIDENCE_THRESHOLD,
         "gazetteer_sha256": hashlib.sha256(gaz.to_csv(index=False).encode()).hexdigest()}
FROZEN_PATH = EVAL_DIR / "frozen_setup.json"
if FROZEN_PATH.exists():
    frozen = json.loads(FROZEN_PATH.read_text())
    assert {k: frozen[k] for k in SETUP} == SETUP, \
        f"Setup changed after the test set was scored (frozen on {frozen['frozen_on']}); the test score no longer applies"
else:
    FROZEN_PATH.write_text(json.dumps({**SETUP, "frozen_on": datetime.now().strftime("%Y-%m-%d")}, indent=2))

test = eval_df[eval_df["split"].eq("test") & in_scope]
test_out = ner(test["text"].tolist(), batch_size=BATCH_SIZE)
test_preds = pd.DataFrame(
    [{"uid": uid, "entity": text[e["start"]:e["end"]], "type": e["entity_group"], "score": float(e["score"]),
      "start": e["start"], "end": e["end"]}
     for uid, text, ents in zip(test["uid"], test["text"], test_out) for e in ents],
    columns=["uid", "entity", "type", "score", "start", "end"])
test_gold   = dict(zip(test["uid"], test["gold_entities"].map(gold_set)))
test_dslim  = pred_sets(CONFIDENCE_THRESHOLD, test_preds)
test_merged = merged_sets(CONFIDENCE_THRESHOLD, test, test_preds)

def bootstrap_f1(gold, pred, n=2_000, seed=SEED):
    # resample sentences with replacement; F1 = 2tp / (2tp + fp + fn)
    counts = np.array([[len(g & pred.get(u, set())), len(pred.get(u, set()) - g), len(g - pred.get(u, set()))]
                       for u, g in gold.items()])
    rng = np.random.default_rng(seed)
    tp, fp, fn = counts[rng.integers(0, len(counts), (n, len(counts)))].sum(axis=1).T
    f1 = np.divide(2 * tp, 2 * tp + fp + fn, out=np.zeros(n), where=tp > 0)
    return np.percentile(f1, [2.5, 97.5])

rows = {"dslim": (test_gold, test_dslim), "dslim + gazetteer": (test_gold, test_merged)}
test_results = pd.DataFrame({name: {**prf(g, p), **dict(zip(["f1_low", "f1_high"], bootstrap_f1(g, p)))}
                             for name, (g, p) in rows.items()}).T
for t in ENT_TYPES:
    test_results.loc[f"dslim + gazetteer: {t}", ["precision", "recall", "f1", "tp", "fp", "fn"]] = \
        pd.Series(prf(test_gold, test_merged, only=t))
for src, part in test.groupby("source"):
    g = {u: test_gold[u] for u in part["uid"]}
    test_results.loc[f"dslim + gazetteer: {src}", ["precision", "recall", "f1", "tp", "fp", "fn"]] = \
        pd.Series(prf(g, test_merged))
print(f"{len(test)} test sentences, {sum(map(len, test_gold.values()))} gold entities")
display(test_results.round(3))

test_errors = pd.DataFrame(
    [{"source": src, "text": text, "missed": fmt(test_gold[uid] - test_merged[uid]),
      "extra": fmt(test_merged[uid] - test_gold[uid])}
     for uid, src, text in zip(test["uid"], test["source"], test["text"]) if test_gold[uid] != test_merged[uid]])
test_results.to_csv(EVAL_DIR / "test_results.csv")
test_errors.to_csv(EVAL_DIR / "test_errors.csv", index=False)
print(f"{len(test_errors)} sentences with errors saved to {EVAL_DIR / 'test_errors.csv'} (for the write-up only)")

100 test sentences, 44 gold entities


,precision,recall,f1,tp,fp,fn,f1_low,f1_high
dslim,0.466,0.614,0.529,27.0,31.0,17.0,0.370,0.679
dslim + gazetteer,0.525,0.727,0.610,32.0,29.0,12.0,0.455,0.752
dslim + gazetteer: PER,0.286,1.000,0.444,2.0,5.0,0.0,NaN,NaN
dslim + gazetteer: ORG,0.852,0.697,0.767,23.0,4.0,10.0,NaN,NaN
dslim + gazetteer: LOC,0.778,0.778,0.778,7.0,2.0,2.0,NaN,NaN
dslim + gazetteer: MISC,0.000,0.000,0.000,0.0,18.0,0.0,NaN,NaN
dslim + gazetteer: everlane_web,0.364,0.889,0.516,8.0,14.0,1.0,NaN,NaN
dslim + gazetteer: gdelt_articles,0.529,0.643,0.581,9.0,8.0,5.0,NaN,NaN
dslim + gazetteer: reddit_comments,0.400,0.444,0.421,4.0,6.0,5.0,NaN,NaN
dslim + gazetteer: reddit_posts,0.917,0.917,0.917,11.0,1.0,1.0,NaN,NaN


22 sentences with errors saved to C:\Users\micha\everlane_deepdive\data\eval\test_errors.csv (for the write-up only)


# 7. Full Corpus

The frozen setup (dslim at 0.80, merged with the gazetteer) now runs on every English sentence. Two files are written to this run's folder:

- `entities.csv`: one row per entity, keyed by `uid`, with the source, date, `date_source` and period attached. `origin` says whether dslim or the gazetteer found it, and `canonical` groups spellings of the same gazetteer brand ("zara" and "Zara" both become "Zara").
- `sentences.csv`: the cleaned sentence table the entities were extracted from, so other notebooks (e.g. sentiment) can join on `uid`.

Based on the test results, the analysis uses ORG and LOC. MISC entities are kept in the file but left out of the analysis (0 of 18 test predictions were correct, mostly website headings and demographic categories), and PER is used with caution (brands below the gazetteer cutoff get tagged as people). This scoping decision was made after scoring test and changes nothing in the setup above.

In [13]:
corpus = df if DEV_SAMPLE_N is None else df.sample(n=DEV_SAMPLE_N, random_state=SEED)
print(f"Running on {len(corpus):,} sentences")
corpus_out = ner(corpus["text"].tolist(), batch_size=BATCH_SIZE)

records = []
for row, ents in zip(corpus.itertuples(), corpus_out):
    model_ents = [(e["start"], e["end"], e["entity_group"], float(e["score"]))
                  for e in ents if e["score"] >= CONFIDENCE_THRESHOLD]
    for s, e, t, score, origin, canonical in merge_entities(row.text, model_ents):
        records.append((row.uid, row.text[s:e], canonical, t, score, origin, s, e))

META = ["uid", "source", "doc_id", "sentence_id", "date", "date_source", "period"]
entities = (pd.DataFrame(records, columns=["uid", "entity", "canonical", "type", "score", "origin", "start", "end"])
              .merge(corpus[META], on="uid", how="left"))
sentences = corpus[META + ["text"]]
entities.to_csv(OUT_DIR / "entities.csv", index=False)
sentences.to_csv(OUT_DIR / "sentences.csv", index=False)
print(f"{len(entities):,} entities from {entities['uid'].nunique():,} sentences -> {OUT_DIR}")

display(pd.crosstab(entities["source"], entities["type"], margins=True))
display(pd.crosstab(entities["type"], entities["origin"]))
top_orgs = (entities[entities["type"].eq("ORG")].groupby("canonical").size()
                                               .sort_values(ascending=False).head(25).rename("mentions"))
display(top_orgs.to_frame().T)

Running on 51,416 sentences


38,716 entities from 21,753 sentences -> C:\Users\micha\everlane_deepdive\data\ner\20260927_0327


type,LOC,MISC,ORG,PER,All
source,,,,,
everlane_web,103,291,407,53,854
gdelt_articles,3373,2826,16592,5538,28329
reddit_comments,728,959,4597,1009,7293
reddit_posts,244,323,1417,256,2240
All,4448,4399,23013,6856,38716


origin,dslim,gazetteer
type,,
LOC,4448,0
MISC,4150,249
ORG,6608,16405
PER,6856,0


canonical,Everlane,Shein,L Catterton,Allbirds,Uniqlo,NPR,Madewell,Gap,Amazon,Reformation,...,Instagram,Banana Republic,Zara,Target,Rothy's,TikTok,Fast Company,Puck News,Kearney Consumer Institute,Birkenstock
mentions,7780,2727,512,435,330,306,250,228,215,204,...,135,123,116,115,112,111,110,105,102,98
